# All you need to modify

In [278]:
import pandas as pd
import os
import numpy as np

# Input file paths
students_sheets = pd.read_excel(
    '/Users/lgteam/Desktop/Weekly Updates Automation Tool/Data/Weekly Reports (Guo) 0504 - CSD6 - Students - Nov 5 2026.xlsx',
    sheet_name=None
)

adults_sheets = '/Users/lgteam/Desktop/Weekly Updates Automation Tool/Data/Weekly Reports (Guo) 0504 - CSD6 - Adults - Nov 5 2026.xlsx'

all_sheets = '/Users/lgteam/Desktop/Weekly Updates Automation Tool/Data/Weekly Reports (Guo) 0504 - CSD6 - All - Nov 5 2026.xlsx'

annual_schedule_path = '/Users/lgteam/Desktop/Weekly Updates Automation Tool/Data/ActivitySessionDetails - CSD6 - Nov 5 2026.xlsx'

missing_attendance_path = "/Users/lgteam/Desktop/Weekly Updates Automation Tool/Data/Gary's Missing Attendance Report - CSD6 - Nov 5 2026.xlsx"

# Student Summary Statistics - Target # of Students Served (don't include Total)
target_values = [125,195,95,160]


/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


# Code Part (don't need to modify)

# Updated info:
1. Created master site list to include sites that have not entered any data to date
2. Included Missing Attendance (%), Staff Assigned, Partner Assigned in the "Site Summary Reports: Attendance by Activity" sheet;
Precaution: 
(a) Partner Assigned col must be manually checked to ensure accuracy as the tool only detects empty entry; 
(b) Color coding must be manually applied for family events, community service learning projects, and summer programming
3. Updadated the adult participation calculation
4. Community Service Summary Statistics needs to be pulled from a separate report
5. Missing Attendance (%) needs to be pulled from a separate report

In [279]:
# Master site list, including sites represented only by placeholder rows.
student_hours = students_sheets["Participant Attendance Hours"].copy()

# Actual headers are in Excel row 3
# (index 1 because read_excel already used row 1 as its header)
student_hours.columns = student_hours.iloc[1]
student_hours = student_hours.iloc[2:].reset_index(drop=True)
student_hours.columns.name = None

# Keep site names, including sites with only placeholder rows
site_names = student_hours["Site"].astype("string").str.strip()

valid_sites = (
    site_names.notna()
    & site_names.ne("")
    & ~site_names.str.lower().isin(["nan", "total", "subtotal"])
)

site_master = (
    site_names.loc[valid_sites]
    .drop_duplicates()
    .rename("Site")
    .to_frame()
    .reset_index(drop=True)
)

if len(site_master) != len(target_values):
    raise ValueError(
        f"Found {len(site_master)} sites but "
        f"{len(target_values)} targets."
    )

site_master["Target # of students served"] = target_values

print(f"Found {len(site_master)} sites. Verify the targets:")
display(site_master)

Found 4 sites. Verify the targets:


,Site,Target # of students served
0,High School for Media and Communications,125
1,P.S. 28 Wright Brothers,195
2,PS 5 Ellen Lurie,95
3,The William Lynch School,160


## 🍀Student Summary Statistics:


In [280]:
# Import the "Participants By Hour Band (Site)" sheet and clean it up
df_part_by_hour = students_sheets[
    "Participants By Hour Band (Site"
].copy()

# Find the first row containing any master-list site
master_sites = site_master["Site"].astype(str).str.strip()

matches = df_part_by_hour.apply(
    lambda col: col.astype(str).str.strip().isin(master_sites)
).any(axis=1)

if not matches.any():
    raise ValueError("No master-list sites found in the hour-band sheet.")

first_site_row = matches[matches].index[0]

if first_site_row == 0:
    raise ValueError("Cannot find a header row above the first site.")

# Use the preceding row as headers, then keep the data rows
df_part_by_hour.columns = (
    df_part_by_hour.iloc[first_site_row - 1]
    .astype("string")
    .str.strip()
)

df_part_by_hour = (
    df_part_by_hour.iloc[first_site_row:]
    .reset_index(drop=True)
)
df_part_by_hour.columns.name = None

if "Site" not in df_part_by_hour.columns:
    raise ValueError("The detected header row does not contain 'Site'.")

display(df_part_by_hour.head(10))

,Institution,Site,Registered Adults,Attended Adults,Registered Youth,Attended Youth,Less Than 15 Hours,15-44 Hours,45-89 Hours,90-179 Hours,180-269 Hours,270+ Hours,15+ Hours,Last Attendance Date
0,"Community School District 6, NYCDOE 8029",High School for Media and Communications,50,0,382,125,114,11,0,0,0,0,11,2026-10-23 00:00:00
1,"Community School District 6, NYCDOE 8029",P.S. 28 Wright Brothers,0,0,155,139,46,93,0,0,0,0,93,2026-10-30 00:00:00
2,"Community School District 6, NYCDOE 8029",PS 5 Ellen Lurie,0,0,90,89,10,79,0,0,0,0,79,2026-11-02 00:00:00
3,"Community School District 6, NYCDOE 8029",The William Lynch School,0,0,276,118,33,47,38,0,0,0,85,2026-10-28 00:00:00
4,NaN,NaN,50,0,903,471,203,230,38,0,0,0,268,NaN


In [281]:
# Keep actual site rows, regardless of Institution
site_names = df_part_by_hour["Site"].astype("string").str.strip()

valid_site = (
    site_names.notna()
    & site_names.ne("")
    & ~site_names.str.lower().isin(["nan", "subtotal", "total"])
)

df_part_by_hour = df_part_by_hour.loc[valid_site].copy()
df_part_by_hour["Site"] = site_names.loc[valid_site]
df_part_by_hour = df_part_by_hour.reset_index(drop=True)

df_part_by_hour

,Institution,Site,Registered Adults,Attended Adults,Registered Youth,Attended Youth,Less Than 15 Hours,15-44 Hours,45-89 Hours,90-179 Hours,180-269 Hours,270+ Hours,15+ Hours,Last Attendance Date
0,"Community School District 6, NYCDOE 8029",High School for Media and Communications,50,0,382,125,114,11,0,0,0,0,11,2026-10-23 00:00:00
1,"Community School District 6, NYCDOE 8029",P.S. 28 Wright Brothers,0,0,155,139,46,93,0,0,0,0,93,2026-10-30 00:00:00
2,"Community School District 6, NYCDOE 8029",PS 5 Ellen Lurie,0,0,90,89,10,79,0,0,0,0,79,2026-11-02 00:00:00
3,"Community School District 6, NYCDOE 8029",The William Lynch School,0,0,276,118,33,47,38,0,0,0,85,2026-10-28 00:00:00


In [282]:
# Get the average  column from Daily Site Attendance Summary
daily_site_att = students_sheets[
    "Daily Site Attendance Summary"
].copy()

master_sites = site_master["Site"].astype(str).str.strip()

matches = daily_site_att.apply(
    lambda col: col.astype(str).str.strip().isin(master_sites)
).any(axis=1)

if not matches.any():
    # No site attendance data: retain an empty table for merging later
    daily_site_att = pd.DataFrame(columns=["Site"])
else:
    first_site_row = matches[matches].index[0]

    if first_site_row == 0:
        raise ValueError("Cannot find a header row above daily attendance.")

    daily_site_att.columns = daily_site_att.iloc[first_site_row - 1]
    daily_site_att = (
        daily_site_att.iloc[first_site_row:]
        .reset_index(drop=True)
    )
    daily_site_att.columns.name = None

    daily_site_att["Site"] = (
        daily_site_att["Site"].astype("string").str.strip()
    )

    daily_site_att = daily_site_att.loc[
        daily_site_att["Site"].isin(master_sites)
    ].copy()

    if daily_site_att["Site"].duplicated().any():
        raise ValueError("Duplicate sites found in daily attendance.")

display(daily_site_att.head())

,Site,2026-09-14 00:00:00,2026-09-15 00:00:00,2026-09-16 00:00:00,2026-09-17 00:00:00,2026-09-18 00:00:00,2026-09-22 00:00:00,2026-09-23 00:00:00,2026-09-24 00:00:00,2026-09-25 00:00:00,...,2026-10-20 00:00:00,2026-10-21 00:00:00,2026-10-22 00:00:00,2026-10-23 00:00:00,2026-10-26 00:00:00,2026-10-27 00:00:00,2026-10-28 00:00:00,2026-10-29 00:00:00,2026-10-30 00:00:00,2026-11-02 00:00:00
0,High School for Media and Communications,0,0,0,0,0,0,0,0,0,...,17,19,22,10,0,0,0,0,0,0
1,P.S. 28 Wright Brothers,0,0,0,0,0,0,0,0,0,...,130,129,128,78,87,98,98,98,97,0
2,PS 5 Ellen Lurie,0,0,0,0,0,0,0,0,0,...,45,33,53,68,52,49,34,49,0,43
3,The William Lynch School,77,88,81,89,50,87,85,73,51,...,84,67,78,0,0,0,96,0,0,0


In [283]:
# Calculate the average # of students served per site across the date cols, rounding to the nearest whole number

from datetime import datetime, date

date_cols = [
    col for col in daily_site_att.columns
    if isinstance(col, (pd.Timestamp, datetime, date))
]

if daily_site_att.empty:
    daily_site_att["Total"] = pd.Series(dtype="Int64")
else:
    if not date_cols:
        raise ValueError("No date columns found in daily attendance.")

    daily_values = daily_site_att[date_cols].apply(
        pd.to_numeric,
        errors="coerce"
    )

    daily_site_att["Total"] = (
        daily_values.mean(axis=1).round().astype("Int64")
    )

daily_site_att = daily_site_att[["Site", "Total"]]

display(daily_site_att)

,Site,Total
0,High School for Media and Communications,11
1,P.S. 28 Wright Brothers,36
2,PS 5 Ellen Lurie,27
3,The William Lynch School,62


In [284]:
print(df_part_by_hour.columns)

Index(['Institution', 'Site', 'Registered Adults', 'Attended Adults',
       'Registered Youth', 'Attended Youth', 'Less Than 15 Hours',
       '15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours',
       '270+ Hours', '15+ Hours', 'Last Attendance Date'],
      dtype='string')


In [285]:
all_cols = ['0 Hours', 'Less Than 15 Hours', '15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours', '270+ Hours']
served_cols = ['Less Than 15 Hours', '15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours', '270+ Hours']
plus15_cols = ['15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours', '270+ Hours']
plus90_cols = ['90-179 Hours', '180-269 Hours', '270+ Hours']

# Filter each col list to only include columns that exist in the dataframe
existing_cols = df_part_by_hour.columns.tolist()
all_cols     = [c for c in all_cols     if c in existing_cols]
served_cols  = [c for c in served_cols  if c in existing_cols]
plus15_cols  = [c for c in plus15_cols  if c in existing_cols]
plus90_cols  = [c for c in plus90_cols  if c in existing_cols]

# Convert hour-band counts to numeric
df_part_by_hour[all_cols] = df_part_by_hour[all_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)

# Enrollment includes students who have not attended
df_part_by_hour["Registered Youth"] = pd.to_numeric(
    df_part_by_hour["Registered Youth"],
    errors="coerce"
).fillna(0)

# Build summary with Site retained for matching
df_totals = pd.DataFrame({
    "Site": df_part_by_hour["Site"],
    "Total # Enrolled": df_part_by_hour["Registered Youth"],
    "Total # Served": df_part_by_hour[served_cols].sum(axis=1),
    "Total # 15+": df_part_by_hour[plus15_cols].sum(axis=1),
    "Total # 90+": df_part_by_hour[plus90_cols].sum(axis=1),
})

# Match targets by site rather than row position
df_totals = site_master[
    ["Site", "Target # of students served"]
].merge(
    df_totals,
    on="Site",
    how="left",
    validate="one_to_one"
)

display(df_totals)

,Site,Target # of students served,Total # Enrolled,Total # Served,Total # 15+,Total # 90+
0,High School for Media and Communications,125,382,125,11,0
1,P.S. 28 Wright Brothers,195,155,139,93,0
2,PS 5 Ellen Lurie,95,90,89,79,0
3,The William Lynch School,160,276,118,85,0


In [286]:
## Add average daily attendance by Site
daily_site_att["Total"] = daily_site_att["Total"].astype("Int64")

daily_site_att = daily_site_att.rename(
    columns={"Total": "Avg. # of Students Per Day"}
)

df_totals = df_totals.merge(
    daily_site_att[["Site", "Avg. # of Students Per Day"]],
    on="Site",
    how="left",
    validate="one_to_one"
)

display(df_totals)

,Site,Target # of students served,Total # Enrolled,Total # Served,Total # 15+,Total # 90+,Avg. # of Students Per Day
0,High School for Media and Communications,125,382,125,11,0,11
1,P.S. 28 Wright Brothers,195,155,139,93,0,36
2,PS 5 Ellen Lurie,95,90,89,79,0,27
3,The William Lynch School,160,276,118,85,0,62


In [287]:
## School names and total row
df_totals = df_totals.rename(columns={"Site": "School"})

df_totals = df_totals[
    [
        "School",
        "Target # of students served",
        "Total # Enrolled",
        "Total # Served",
        "Avg. # of Students Per Day",
        "Total # 15+",
        "Total # 90+",
    ]
]

# Sum numeric columns
total_row = df_totals.iloc[:, 1:].sum(min_count=1).to_frame().T
total_row.insert(0, "School", "Total")

# Avoid presenting a partial attendance sum as the complete grant total
if df_totals["Avg. # of Students Per Day"].isna().any():
    total_row["Avg. # of Students Per Day"] = pd.NA

df_totals = pd.concat(
    [df_totals, total_row],
    ignore_index=True
)

display(df_totals)

,School,Target # of students served,Total # Enrolled,Total # Served,Avg. # of Students Per Day,Total # 15+,Total # 90+
0,High School for Media and Communications,125,382,125,11,11,0
1,P.S. 28 Wright Brothers,195,155,139,36,93,0
2,PS 5 Ellen Lurie,95,90,89,27,79,0
3,The William Lynch School,160,276,118,62,85,0
4,Total,575,903,471,136,268,0


In [288]:
# Format counts and percentages
for hours in [15, 90]:
    count_col = f"Total # {hours}+"
    display_col = f"# of students {hours}+ hrs total (% of Target)"

    if count_col in df_totals.columns:
        counts = pd.to_numeric(
            df_totals[count_col], errors="coerce"
        ).astype("Int64")

        targets = pd.to_numeric(
            df_totals["Target # of students served"],
            errors="coerce"
        ).astype("Float64")

        percentages = (
            counts.div(targets.where(targets > 0))
            .mul(100)
            .round()
            .astype("Int64")
        )

        df_totals[display_col] = (
            counts.astype("string").fillna("-")
            + " ("
            + percentages.astype("string").fillna("-")
            + "%)"
        )

        df_totals.loc[counts.isna(), display_col] = "-"

    elif display_col not in df_totals.columns:
        raise ValueError(
            f"Missing {count_col}. Rerun the student-count cells first."
        )

# Remove raw count columns if present
df_totals = df_totals.drop(
    columns=["Total # 15+", "Total # 90+"],
    errors="ignore"
).reset_index(drop=True)

# Keep numeric counts as integers, allowing missing values
count_columns = [
    "Target # of students served",
    "Total # Enrolled",
    "Total # Served",
    "Avg. # of Students Per Day",
]

for col in count_columns:
    df_totals[col] = pd.to_numeric(
        df_totals[col], errors="coerce"
    ).astype("Int64")


# Color valid 15+ percentages
def color_pct_cols(val):
    try:
        pct = int(str(val).split("(")[1].replace("%)", "").strip())
        return "color: green" if pct >= 100 else "color: red"
    except (ValueError, IndexError):
        return ""


styled_totals = (
    df_totals.style
    .apply(
        lambda col: col.map(color_pct_cols),
        subset=["# of students 15+ hrs total (% of Target)"]
    )
    .format(
        {col: "{:.0f}" for col in count_columns},
        na_rep="-"
    )
)

display(styled_totals)

,School,Target # of students served,Total # Enrolled,Total # Served,Avg. # of Students Per Day,# of students 15+ hrs total (% of Target),# of students 90+ hrs total (% of Target)
0,High School for Media and Communications,125,382,125,11,11 (9%),0 (0%)
1,P.S. 28 Wright Brothers,195,155,139,36,93 (48%),0 (0%)
2,PS 5 Ellen Lurie,95,90,89,27,79 (83%),0 (0%)
3,The William Lynch School,160,276,118,62,85 (53%),0 (0%)
4,Total,575,903,471,136,268 (47%),0 (0%)


## 🌷Family Component Summary Statistics (Last Column)

In [289]:
df_hours = pd.read_excel(
    adults_sheets,
    sheet_name="Participant Attendance Hours",
    header=None
)

header_text = df_hours.apply(
    lambda col: col.astype("string").str.strip()
)

header_matches = (
    header_text.eq("Site").any(axis=1)
    & header_text.eq("Participant Id").any(axis=1)
    & header_text.eq("Hours Present").any(axis=1)
)

if not header_matches.any():
    raise ValueError(
        "Cannot find Site, Participant Id, and Hours Present headers."
    )

header_row = header_matches[header_matches].index[0]

df_hours.columns = header_text.iloc[header_row]
df_hours = df_hours.iloc[header_row + 1:].reset_index(drop=True)
df_hours.columns.name = None

print(f"Header detected in Excel row {header_row + 1}.")
display(df_hours.head())

Header detected in Excel row 3.


/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Institution,Site,Last Name,First Name,Participant Id,State Participant Id,School,Grade Level,Teachers,Days Present,Hours Present,Hour Band
0,"Community School District 6, NYCDOE 8029",High School for Media and Communications,Angeles,Alexandra,1779383617704723AA,1779383617704723AA,NaN,NaN,NaN,0,0,0
1,"Community School District 6, NYCDOE 8029",High School for Media and Communications,Anliezo,Olego,1779387630771841OA,1779387630771841OA,NaN,NaN,NaN,0,0,0
2,"Community School District 6, NYCDOE 8029",High School for Media and Communications,Belthe,Rossy,1779386220622783RB,1779386220622783RB,NaN,NaN,NaN,0,0,0
3,"Community School District 6, NYCDOE 8029",High School for Media and Communications,Bemoy,Heidi,1779386584929833HB,1779386584929833HB,NaN,NaN,NaN,0,0,0
4,"Community School District 6, NYCDOE 8029",High School for Media and Communications,Bravo,Jeff,1779384481545051JB,1779384481545051JB,NaN,NaN,NaN,0,0,0


In [290]:
df_hours["Site"] = (
    df_hours["Site"].astype("string").str.strip()
)

df_hours["Hours Present"] = pd.to_numeric(
    df_hours["Hours Present"], errors="coerce"
)

df_hours["Participant Id"] = (
    df_hours["Participant Id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

valid_id = (
    df_hours["Participant Id"].notna()
    & df_hours["Participant Id"].ne("")
    & ~df_hours["Participant Id"].str.lower().isin(["nan", "none"])
)

df_active = df_hours.loc[
    (df_hours["Hours Present"] > 0)
    & valid_id
    & df_hours["Participant Id"].str.len().ne(9)
].copy()

# Check that qualifying participants have recognized sites
if df_active["Site"].isna().any() or df_active["Site"].eq("").any():
    raise ValueError("An adult with attendance has no Site.")

unexpected_sites = (
    set(df_active["Site"]) - set(site_master["Site"])
)

if unexpected_sites:
    raise ValueError(
        f"Adult sites missing from master list: {unexpected_sites}"
    )

print(f"Qualifying attendance records: {len(df_active)}")

Qualifying attendance records: 0


In [291]:
result = (
    df_active.groupby("Site")["Participant Id"]
    .nunique()
    .reset_index(name="Parents Served (Total)")
)

result = site_master[["Site"]].merge(
    result,
    on="Site",
    how="left",
    validate="one_to_one"
)

result["Parents Served (Total)"] = (
    result["Parents Served (Total)"].fillna(0).astype(int)
)

result.loc[len(result)] = {
    "Site": "Total",
    "Parents Served (Total)": result["Parents Served (Total)"].sum()
}

display(result)

,Site,Parents Served (Total)
0,High School for Media and Communications,0
1,P.S. 28 Wright Brothers,0
2,PS 5 Ellen Lurie,0
3,The William Lynch School,0
4,Total,0


##  🌸Participant Demographics - missing info

In [292]:
# Load the Participant Demographics sheet from the students workbook and clean it up
df_part_demo = students_sheets["Participant Demographics"].copy()

# If headers are already correct, no detection is needed
if not {"Site", "Date Of Birth", "Grade Level"}.issubset(
    df_part_demo.columns
):
    header_text = df_part_demo.apply(
        lambda col: col.astype("string").str.strip()
    )

    header_matches = (
        header_text.eq("Site").any(axis=1)
        & header_text.eq("Date Of Birth").any(axis=1)
        & header_text.eq("Grade Level").any(axis=1)
    )

    if not header_matches.any():
        raise ValueError(
            "Cannot find demographics headers. "
            "Rerun the original import cell to reload students_sheets."
        )

    header_row = header_matches[header_matches].index[0]

    df_part_demo.columns = header_text.iloc[header_row]
    df_part_demo = (
        df_part_demo.iloc[header_row + 1:]
        .reset_index(drop=True)
    )

df_part_demo.columns = df_part_demo.columns.astype(str).str.strip()
df_part_demo.columns.name = None

# Validate expected columns
required_columns = [
    "Site",
    "Last Name",
    "First Name",
    "ParticipantID",
    "State ParticipantID",
    "Date Of Birth",
    "Grade Level",
    "Gender",
]

missing_columns = [
    col for col in required_columns
    if col not in df_part_demo.columns
]

if missing_columns:
    raise ValueError(
        f"Missing demographics columns: {missing_columns}"
    )

# Exclude empty placeholders; retain students whose ID is missing
identity_columns = ["Last Name", "First Name", "ParticipantID"]

has_identity = df_part_demo[identity_columns].apply(
    lambda col: (
        col.astype("string").str.strip().fillna("").ne("")
    )
).any(axis=1)

df_part_demo = df_part_demo.loc[has_identity].reset_index(drop=True)

print(f"Participant records loaded: {len(df_part_demo)}")
display(df_part_demo.head())

Participant records loaded: 903


,Institution,Site,Last Name,First Name,ParticipantID,State ParticipantID,Adult,Registration Start Date,Registration End Date,Currently Active,Date Of Birth,LEA,School,Grade Level,Gender,Race/Ethnicity,English Learner Status,Lunch Status,Special Education Status,IDEA Disability Type
0,"Community School District 6, NYCDOE 8029",The William Lynch School,Abraham,Ella,253092761,4.983550e+09,No,2026-07-01,2027-06-30,Yes,2017-04-09,"Community School District 6, NYCDOE 8029",The William Lynch School,11,Female,Race and Ethnicity Unknown,No,Reduced price,No,Not Entered
1,"Community School District 6, NYCDOE 8029",The William Lynch School,Abraham,Zachary,250131188,2.501312e+08,No,2026-07-01,2027-06-30,Yes,2021-12-29,"Community School District 6, NYCDOE 8029",The William Lynch School,KG,Male,White,No,Reduced price,No,Not Entered
2,"Community School District 6, NYCDOE 8029",P.S. 28 Wright Brothers,Acevedo,Athziri,257510206,7.482367e+09,No,2026-07-01,2027-06-30,Yes,2019-10-17,"Community School District 6, NYCDOE 8029",P.S. 28 Wright Brothers,Not Entered,Female,Hispanic or Latino,No,Free,No,Not Entered
3,"Community School District 6, NYCDOE 8029",The William Lynch School,Acosta,Alaia,257091165,1.879809e+09,No,2026-07-01,2027-06-30,Yes,2018-07-21,"Community School District 6, NYCDOE 8029",The William Lynch School,03,Female,Hispanic or Latino,Yes,Reduced price,No,Not Entered
4,"Community School District 6, NYCDOE 8029",High School for Media and Communications,Adam,Ayisha,258299999,4.459361e+09,No,2026-07-01,2027-06-30,Yes,2006-12-13,"Community School District 6, NYCDOE 8029",High School for Media and Communications,11,Female,Race and Ethnicity Unknown,No,Reduced price,No,Not Entered


In [293]:
# Flag missing required demo info
all_missing_flags = df_part_demo.copy()

def clean_text(series):
    return series.astype("string").str.strip()

def is_missing(series):
    text = clean_text(series)
    return (
        text.isna()
        | text.str.lower().isin(
            ["", "not entered", "nan", "none"]
        )
    )

# Clean site names
all_missing_flags["Site"] = clean_text(all_missing_flags["Site"])
all_missing_flags.loc[
    is_missing(all_missing_flags["Site"]), "Site"
] = pd.NA

# Clean IDs
for col in ["ParticipantID", "State ParticipantID"]:
    all_missing_flags[col] = (
        clean_text(all_missing_flags[col])
        .str.replace(r"\.0$", "", regex=True)
    )

# Missing or unparseable DOB
dob_parsed = pd.to_datetime(
    all_missing_flags["Date Of Birth"],
    errors="coerce"
)

all_missing_flags["Date Of Birth_missing"] = (
    dob_parsed.isna().astype(int)
)

# Required participant ID: nine digits starting with 1 or 2
all_missing_flags["ParticipantID_missing"] = (
    ~all_missing_flags["ParticipantID"].str.fullmatch(
        r"[12]\d{8}", na=False
    )
).astype(int)

all_missing_flags["Grade Level_missing"] = (
    is_missing(all_missing_flags["Grade Level"]).astype(int)
)

all_missing_flags["Gender_missing"] = (
    ~clean_text(all_missing_flags["Gender"])
    .str.lower()
    .isin(["male", "female", "non-binary"])
).astype(int)

# Optional state ID: ten digits
all_missing_flags["State ParticipantID_missing"] = (
    ~all_missing_flags["State ParticipantID"].str.fullmatch(
        r"\d{10}", na=False
    )
).astype(int)

In [294]:
# Flag unusual age for grade
# Allow for students who are up to 3 years older or younger than typical for their grade;
# Allow for students 1-9 years old for PK/KG

report_date = pd.Timestamp.today().normalize()
age_tolerance = 3

grade_text = clean_text(
    all_missing_flags["Grade Level"]
).str.upper()

grade_number = pd.to_numeric(grade_text, errors="coerce")

# Recognize PK and kindergarten labels
is_pk_kg = grade_text.isin([
    "PK", "PRE-K", "PREK", "PRE-KINDERGARTEN",
    "K", "KG", "KN", "KINDERGARTEN"
]) | grade_number.eq(0)

# Apply typical-age screening only to Grades 1–12
recognized_grade = (
    grade_number.between(1, 12)
    & grade_number.mod(1).eq(0)
).fillna(False)

typical_age = (grade_number + 5).where(recognized_grade)

# Age in completed years
birthday_not_reached = (
    (dob_parsed.dt.month > report_date.month)
    | (
        (dob_parsed.dt.month == report_date.month)
        & (dob_parsed.dt.day > report_date.day)
    )
)

age = (
    report_date.year
    - dob_parsed.dt.year
    - birthday_not_reached.astype(int)
)

# PK/KG: allow ages 1–9 inclusive
pk_kg_mismatch = (
    is_pk_kg
    & dob_parsed.notna()
    & (age.lt(1) | age.gt(9))
).fillna(False)

# Grades 1–12: allow typical age ±3 inclusive
age_grade_mismatch = (
    recognized_grade
    & dob_parsed.notna()
    & (age - typical_age).abs().gt(age_tolerance)
).fillna(False)

future_dob = dob_parsed.gt(report_date)

all_missing_flags["DOB_age_grade_review"] = (
    pk_kg_mismatch | age_grade_mismatch | future_dob
).astype(int)

dob_review_details = pd.DataFrame({
    "Age at Report Date": age.astype("Int64"),
    "Typical Age for Grade": typical_age.astype("Int64"),
    "Review Reason": np.select(
        [
            future_dob.to_numpy(dtype=bool),
            pk_kg_mismatch.to_numpy(dtype=bool),
            age_grade_mismatch.to_numpy(dtype=bool),
        ],
        [
            "DOB is in the future",
            "PK/KG age outside 1–9 years",
            "Age more than 3 years from typical grade age",
        ],
        default=""
    ),
})

unrecognized_grades = grade_text.loc[
    ~recognized_grade
    & ~is_pk_kg
    & ~is_missing(all_missing_flags["Grade Level"])
].drop_duplicates().tolist()

print("Grades not included in age screening:", unrecognized_grades)
print(
    "DOB review records:",
    all_missing_flags["DOB_age_grade_review"].sum()
)

Grades not included in age screening: ['13']
DOB review records: 76


In [295]:
# Create Missing Student Summary
summary_flags = [
    "Date Of Birth_missing",
    "ParticipantID_missing",
    "Grade Level_missing",
    "Gender_missing",
    "State ParticipantID_missing",
]

unexpected_sites = (
    set(all_missing_flags["Site"].dropna())
    - set(site_master["Site"])
)

if unexpected_sites:
    raise ValueError(
        f"Demographics sites missing from master list: {unexpected_sites}"
    )

missing_summary = (
    all_missing_flags.groupby("Site")[summary_flags]
    .sum()
    .reset_index()
)

missing_summary = site_master[["Site"]].merge(
    missing_summary,
    on="Site",
    how="left",
    validate="one_to_one"
)

missing_summary[summary_flags] = (
    missing_summary[summary_flags].fillna(0).astype(int)
)

total_row = {"Site": "Total"}
total_row.update(missing_summary[summary_flags].sum().to_dict())

missing_summary = pd.concat(
    [missing_summary, pd.DataFrame([total_row])],
    ignore_index=True
)

missing_summary = missing_summary.rename(columns={
    "Date Of Birth_missing": "DOB_missing",
    "State ParticipantID_missing":
        "10digit_State ParticipantID_missing",
})

display(missing_summary)

,Site,DOB_missing,ParticipantID_missing,Grade Level_missing,Gender_missing,10digit_State ParticipantID_missing
0,High School for Media and Communications,0,0,3,0,76
1,P.S. 28 Wright Brothers,0,0,3,0,49
2,PS 5 Ellen Lurie,0,0,0,0,38
3,The William Lynch School,0,10,0,0,52
4,Total,0,10,6,0,215


In [296]:
# Create student pull-outs
flag_cols = [
    "Date Of Birth_missing",
    "ParticipantID_missing",
    "Grade Level_missing",
    "Gender_missing",
]

total_missing_rows = all_missing_flags.loc[
    all_missing_flags[flag_cols].sum(axis=1) > 0
].copy()

missing_site_rows = all_missing_flags.loc[
    all_missing_flags["Site"].isna()
].copy()

review_mask = all_missing_flags["DOB_age_grade_review"].eq(1)

young_dob_rows = all_missing_flags.loc[review_mask].copy()

young_dob_rows = young_dob_rows.join(
    dob_review_details.loc[review_mask]
)

print(f"Required-info review records: {len(total_missing_rows)}")
print(f"Records without a site: {len(missing_site_rows)}")
print(f"DOB/grade review records: {len(young_dob_rows)}")

Required-info review records: 16
Records without a site: 0
DOB/grade review records: 76


## Missing Staff Summary

In [297]:
# Load the Staff Details sheet again to detect the header row and validate required columns

df_staff = pd.read_excel(
    all_sheets,
    sheet_name="Staff Details",
    header=None
)

header_text = df_staff.apply(
    lambda col: col.astype("string").str.strip()
)

header_matches = (
    header_text.eq("Site").any(axis=1)
    & header_text.eq("Staff Type").any(axis=1)
    & header_text.eq("Compensation Type").any(axis=1)
)

if not header_matches.any():
    raise ValueError("Cannot find the Staff Details header row.")

header_row = header_matches[header_matches].index[0]

df_staff.columns = header_text.iloc[header_row]
df_staff = df_staff.iloc[header_row + 1:].reset_index(drop=True)
df_staff.columns.name = None

required_columns = [
    "Site", "Email Address", "First Name", "Last Name",
    "Staff Type", "Compensation Type", "Funder",
]

missing_columns = [
    col for col in required_columns
    if col not in df_staff.columns
]

if missing_columns:
    raise ValueError(f"Missing staff columns: {missing_columns}")

print(f"Header detected in Excel row {header_row + 1}.")

Header detected in Excel row 3.


/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [298]:
df = df_staff.copy()

# Normalize text used in the checks
text_columns = [
    "Site", "Email Address", "First Name", "Last Name",
    "Staff Type", "Compensation Type", "Funder",
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip().fillna("")
    df.loc[
        df[col].str.lower().isin(["nan", "none", "not entered"]),
        col
    ] = ""

# Exclude site placeholders without any staff identity
has_identity = df[
    ["Email Address", "First Name", "Last Name"]
].ne("").any(axis=1)

df = df.loc[has_identity].copy()

if df["Site"].eq("").any():
    raise ValueError("A staff record has no Site assigned.")

unexpected_sites = set(df["Site"]) - set(site_master["Site"])

if unexpected_sites:
    raise ValueError(
        f"Staff sites missing from master list: {unexpected_sites}"
    )

# Email preferred; otherwise use first and last name
email = df["Email Address"].str.lower()
name_id = (
    df["First Name"].str.lower()
    + "|"
    + df["Last Name"].str.lower()
)

df["_staff_id"] = np.where(
    email.ne(""),
    "email:" + email,
    "name:" + name_id
)

# Preserve the original rule: retain the first record per staff/site
df = df.drop_duplicates(subset=["Site", "_staff_id"]).copy()

staff_type = df["Staff Type"].str.lower()
comp_type = df["Compensation Type"].str.lower()
funder = df["Funder"]

df["Staff Type_missing"] = (
    staff_type.isin(["", "other"])
).astype(int)

df["Employment Type_missing"] = (
    ~comp_type.isin(["paid", "volunteer"])
).astype(int)

# Preserve existing funding rule:
# non-volunteers need a funder; any entered funder must contain "21"
is_volunteer = comp_type.eq("volunteer")
funder_blank = funder.eq("")
funder_not_21cclc = ~funder.str.contains("21", case=False, na=False)

df["Funded by 21st CCLC_missing"] = (
    (~is_volunteer & funder_blank)
    | (~funder_blank & funder_not_21cclc)
).astype(int)

staff_flag_cols = [
    "Staff Type_missing",
    "Employment Type_missing",
    "Funded by 21st CCLC_missing",
]

staff_missing_rows = df.loc[
    df[staff_flag_cols].sum(axis=1) > 0
].copy()

In [299]:
# Create a summary of missing staff info by site
staff_missing_summary = (
    df.groupby("Site")
    .agg(
        **{
            "# of Active Program Staff": ("_staff_id", "nunique"),
            "Staff Type": ("Staff Type_missing", "sum"),
            "Employment Type": ("Employment Type_missing", "sum"),
            "Funded by 21st CCLC": (
                "Funded by 21st CCLC_missing", "sum"
            ),
        }
    )
    .reset_index()
)

staff_missing_summary = site_master[["Site"]].merge(
    staff_missing_summary,
    on="Site",
    how="left",
    validate="one_to_one"
)

summary_columns = [
    "# of Active Program Staff",
    "Staff Type",
    "Employment Type",
    "Funded by 21st CCLC",
]

staff_missing_summary[summary_columns] = (
    staff_missing_summary[summary_columns].fillna(0).astype(int)
)

total_row = {"Site": "Total"}
total_row.update(
    staff_missing_summary[summary_columns].sum().to_dict()
)

staff_missing_summary = pd.concat(
    [staff_missing_summary, pd.DataFrame([total_row])],
    ignore_index=True
)

display(staff_missing_summary)

,Site,# of Active Program Staff,Staff Type,Employment Type,Funded by 21st CCLC
0,High School for Media and Communications,19,5,0,0
1,P.S. 28 Wright Brothers,25,0,0,0
2,PS 5 Ellen Lurie,11,1,0,0
3,The William Lynch School,11,0,0,0
4,Total,66,6,0,0


## 🪻Site Summary Report

In [300]:
# 1. Load session reports and detect their header rows

def read_session_report(sheet_name):
    raw = pd.read_excel(
        all_sheets,
        sheet_name=sheet_name,
        header=None,
    )

    header_text = raw.apply(
        lambda col: col.astype("string").str.strip()
    )

    header_matches = (
        header_text.eq("Site").any(axis=1)
        & header_text.eq("Activity").any(axis=1)
        & header_text.eq("Session").any(axis=1)
    )

    if not header_matches.any():
        raise ValueError(
            f"Cannot find Site, Activity, and Session headers "
            f"in '{sheet_name}'."
        )

    header_row = header_matches[header_matches].index[0]

    headers = raw.iloc[header_row].map(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

    df = raw.iloc[header_row + 1:].copy()
    df.columns = headers
    df.columns.name = None

    # Remove columns without headers
    df = df.loc[
        :,
        df.columns.notna() & (df.columns != ""),
    ]

    if df.columns.duplicated().any():
        duplicated_headers = (
            df.columns[df.columns.duplicated()]
            .tolist()
        )

        raise ValueError(
            f"Duplicate headers in '{sheet_name}': "
            f"{duplicated_headers}"
        )

    # Clean common session identifiers
    for col in ["Site", "Activity", "Session"]:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
        )

    # Remove blank, total, and subtotal rows
    df = df.loc[
        df["Site"].notna()
        & df["Site"].ne("")
        & ~df["Site"].str.lower().isin(
            ["total", "subtotal", "average"]
        )
    ].copy()

    unexpected_sites = (
        set(df["Site"])
        - set(site_master["Site"])
    )

    if unexpected_sites:
        raise ValueError(
            f"Sites missing from the master list in "
            f"'{sheet_name}': {unexpected_sites}"
        )

    return df.reset_index(drop=True)


df_act = read_session_report(
    "Activity-Session Details"
)

df_enr = read_session_report(
    "Session Enrollment by Session"
)

df_att = read_session_report(
    "Daily Activity Attendance Summa"
)

print("Activity/session rows:", len(df_act))
print("Enrollment rows:", len(df_enr))
print("Attendance rows:", len(df_att))

/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Activity/session rows: 173
Enrollment rows: 165
Attendance rows: 159


In [301]:
# 2. Prepare Activity-Session Details

from datetime import datetime, time
from numbers import Real


def clean_session_time(value):
    """Remove the artificial date and retain a consistent time value."""
    if pd.isna(value):
        return pd.NA

    if isinstance(value, time):
        return value.strftime("%H:%M:%S")

    # Handle Excel time values stored as fractions of a day
    if isinstance(value, Real):
        if 0 <= value < 1:
            seconds = int(round(float(value) * 86400)) % 86400
            return (
                f"{seconds // 3600:02d}:"
                f"{seconds % 3600 // 60:02d}:"
                f"{seconds % 60:02d}"
            )
        raise ValueError(f"Unexpected numeric session time: {value}")

    text = str(value).strip()

    if text.lower() in ["", "nan", "none", "not entered", "-"]:
        return pd.NA

    parsed = pd.to_datetime(text, errors="coerce")

    if pd.isna(parsed):
        raise ValueError(f"Cannot read session time: {value}")

    return parsed.strftime("%H:%M:%S")


cols_act = [
    "Site",
    "Activity",
    "Session",
    "Session Start Date",
    "Session End Date",
    "Session Start Time",
    "Session End Time",
    "Days Scheduled",
    "Designated Staff",
    "Partners",
]

missing_cols = [
    col for col in cols_act
    if col not in df_act.columns
]

if missing_cols:
    raise ValueError(
        f"Missing Activity-Session Details columns: {missing_cols}"
    )

df_act = df_act[cols_act].copy()

for col in ["Session Start Date", "Session End Date"]:
    df_act[col] = pd.to_datetime(
        df_act[col], errors="coerce"
    ).dt.normalize()

for col in ["Session Start Time", "Session End Time"]:
    df_act[col] = df_act[col].map(clean_session_time).astype("string")

df_act["Days Scheduled"] = pd.to_numeric(
    df_act["Days Scheduled"], errors="coerce"
).astype("Int64")

df_act = df_act.rename(columns={
    "Days Scheduled": "Days Scheduled to Date",
    "Designated Staff": "Staff Assigned",
    "Partners": "Partner",
})

for col in ["Staff Assigned", "Partner"]:
    df_act[col] = df_act[col].astype("string").str.strip()

    df_act.loc[
        df_act[col].str.lower().isin(
            ["", "not entered", "nan", "none"]
        ),
        col
    ] = pd.NA

df_act["Partner Assigned"] = np.where(
    df_act["Partner"].notna(), "Y", "N"
)

df_act["Staff Assigned"] = np.where(
    df_act["Staff Assigned"].notna(), "Y", "N"
)

df_act = df_act.drop_duplicates().reset_index(drop=True)

today = (
    pd.Timestamp.now("America/New_York")
    .normalize()
    .tz_localize(None)
)

display(df_act.head())

,Site,Activity,Session,Session Start Date,Session End Date,Session Start Time,Session End Time,Days Scheduled to Date,Staff Assigned,Partner,Partner Assigned
0,High School for Media and Communications,Arts,Dominos,2026-10-02,2027-05-28,16:30:00,17:30:00,6,Y,Arete,Y
1,High School for Media and Communications,Arts,Flower Design,2026-09-29,2027-05-28,15:00:00,17:00:00,11,Y,Arete,Y
2,High School for Media and Communications,Arts,Media Club,2026-09-28,2027-05-28,14:30:00,15:30:00,12,N,<NA>,N
3,High School for Media and Communications,Arts,Printmaking,2026-09-28,2027-05-28,15:00:00,17:00:00,11,Y,<NA>,N
4,High School for Media and Communications,ELA and Math enrichment activities,ELA Support,2026-09-30,2027-06-30,14:30:00,16:30:00,12,Y,<NA>,N


In [302]:
# 3. Prepare Session Enrollment by Session

cols_enr = [
    "Site",
    "Activity",
    "Session",
    "Enrolled Count",
]

missing_cols = [
    col for col in cols_enr
    if col not in df_enr.columns
]

if missing_cols:
    raise ValueError(
        "Missing columns from Session Enrollment by Session: "
        f"{missing_cols}"
    )

df_enr = df_enr[cols_enr].copy()

df_enr["Enrolled Count"] = pd.to_numeric(
    df_enr["Enrolled Count"],
    errors="coerce",
)

df_enr = df_enr.rename(columns={
    "Enrolled Count":
        "Enrolled Participant",
})

# Remove completely identical rows
df_enr = (
    df_enr
    .drop_duplicates()
    .reset_index(drop=True)
)

display(df_enr.head())

,Site,Activity,Session,Enrolled Participant
0,High School for Media and Communications,Arts,Dominos,4
1,High School for Media and Communications,Arts,Flower Design,15
2,High School for Media and Communications,Arts,Media Club,18
3,High School for Media and Communications,Arts,Printmaking,2
4,High School for Media and Communications,ELA and Math enrichment activities,ELA Support,7


In [303]:
# 4. Prepare Daily Activity Attendance Summary

cols_att = [
    "Site",
    "Activity",
    "Session",
    "Total",
]

missing_cols = [
    col for col in cols_att
    if col not in df_att.columns
]

if missing_cols:
    raise ValueError(
        "Missing columns from Daily Activity Attendance Summary: "
        f"{missing_cols}"
    )

df_att = df_att[cols_att].copy()

# Handle numeric values and text such as "Average: 12.5"
attendance_text = (
    df_att["Total"]
    .astype("string")
    .str.replace(
        r"(?i)^average\s*:\s*",
        "",
        regex=True,
    )
    .str.strip()
)

df_att["Total"] = (
    pd.to_numeric(
        attendance_text,
        errors="coerce",
    )
    .round()
    .astype("Int64")
)

df_att = df_att.rename(columns={
    "Total":
        "Average Daily Attendance",
})

# Remove completely identical rows
df_att = (
    df_att
    .drop_duplicates()
    .reset_index(drop=True)
)

display(df_att.head())

,Site,Activity,Session,Average Daily Attendance
0,High School for Media and Communications,Arts,Dominos,4
1,High School for Media and Communications,Arts,Flower Design,7
2,High School for Media and Communications,Arts,Media Club,5
3,High School for Media and Communications,Arts,Printmaking,2
4,High School for Media and Communications,ELA and Math enrichment activities,ELA Support,2


In [304]:
# 5. Define session keys and merge full-year scheduled days

import io

base_keys = ["Site", "Activity", "Session"]

session_keys = base_keys + [
    "Session Start Date",
    "Session Start Time",
]

sites = site_master["Site"].tolist()


def read_external_report(source, sheet_name, required_columns):
    excel_source = (
        io.BytesIO(source)
        if isinstance(source, bytes)
        else source
    )

    raw = pd.read_excel(
        excel_source,
        sheet_name=sheet_name,
        header=None
    )

    header_text = raw.apply(
        lambda col: col.astype("string").str.strip()
    )

    header_matches = pd.Series(True, index=raw.index)

    for col in required_columns:
        header_matches &= header_text.eq(col).any(axis=1)

    if not header_matches.any():
        raise ValueError(
            f"Cannot find headers in '{sheet_name}': "
            f"{required_columns}"
        )

    header_row = header_matches[header_matches].index[0]

    headers = raw.iloc[header_row].map(
        lambda value: value.strip()
        if isinstance(value, str) else value
    )

    df = raw.iloc[header_row + 1:].copy()
    df.columns = headers
    df.columns.name = None

    df = df.loc[
        :, df.columns.notna() & (df.columns != "")
    ]

    if df.columns.duplicated().any():
        raise ValueError(f"Duplicate headers in '{sheet_name}'.")

    df = df[required_columns].copy()

    for col in base_keys:
        df[col] = df[col].astype("string").str.strip()

    df = df.loc[
        df["Site"].notna()
        & df["Site"].ne("")
        & ~df["Site"].str.lower().isin(
            ["total", "subtotal", "average", "nan"]
        )
    ].copy()

    unexpected_sites = set(df["Site"]) - set(site_master["Site"])

    if unexpected_sites:
        raise ValueError(
            f"Sites missing from master list in '{sheet_name}': "
            f"{unexpected_sites}"
        )

    return df.reset_index(drop=True)


# Validate the current activity report using the expanded key
duplicates = df_act.duplicated(session_keys, keep=False)

if duplicates.any():
    display(
        df_act.loc[duplicates].sort_values(session_keys)
    )
    raise ValueError(
        "Duplicate sessions remain after including start date "
        "and start time. Review the displayed records."
    )

# These two reports still have only the three shared identifiers
for report_name, report_df in [
    ("Session Enrollment by Session", df_enr),
    ("Daily Activity Attendance Summary", df_att),
]:
    duplicates = report_df.duplicated(base_keys, keep=False)

    if duplicates.any():
        display(
            report_df.loc[duplicates].sort_values(base_keys)
        )
        raise ValueError(
            f"Duplicate base session keys in {report_name}."
        )

# Load full-year scheduled days with start date AND start time
df_annual = read_external_report(
    annual_schedule_path,
    "Activity-Session Details",
    session_keys + ["Days Scheduled"]
)

df_annual["Session Start Date"] = pd.to_datetime(
    df_annual["Session Start Date"],
    errors="coerce"
).dt.normalize()

df_annual["Session Start Time"] = (
    df_annual["Session Start Time"]
    .map(clean_session_time)
    .astype("string")
)

df_annual["Days Scheduled"] = pd.to_numeric(
    df_annual["Days Scheduled"], errors="coerce"
).astype("Int64")

df_annual = df_annual.drop_duplicates().reset_index(drop=True)

duplicates = df_annual.duplicated(session_keys, keep=False)

if duplicates.any():
    display(
        df_annual.loc[duplicates].sort_values(session_keys)
    )
    raise ValueError(
        "Conflicting annual schedule rows remain after "
        "including start date and start time."
    )

df_act = df_act.merge(
    df_annual,
    on=session_keys,
    how="left",
    validate="one_to_one",
    indicator="_annual_match"
)

unmatched = df_act["_annual_match"].eq("left_only")

if unmatched.any():
    print("Sessions without a matching full-year schedule:")
    display(df_act.loc[unmatched, session_keys])

df_act = df_act.drop(columns="_annual_match")

print(
    f"Full-year scheduled days added for "
    f"{df_act['Days Scheduled'].notna().sum()} "
    f"of {len(df_act)} sessions."
)

display(df_act.head())

Full-year scheduled days added for 173 of 173 sessions.


/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Site,Activity,Session,Session Start Date,Session End Date,Session Start Time,Session End Time,Days Scheduled to Date,Staff Assigned,Partner,Partner Assigned,Days Scheduled
0,High School for Media and Communications,Arts,Dominos,2026-10-02,2027-05-28,16:30:00,17:30:00,6,Y,Arete,Y,36
1,High School for Media and Communications,Arts,Flower Design,2026-09-29,2027-05-28,15:00:00,17:00:00,11,Y,Arete,Y,70
2,High School for Media and Communications,Arts,Media Club,2026-09-28,2027-05-28,14:30:00,15:30:00,12,N,<NA>,N,71
3,High School for Media and Communications,Arts,Printmaking,2026-09-28,2027-05-28,15:00:00,17:00:00,11,Y,<NA>,N,70
4,High School for Media and Communications,ELA and Math enrichment activities,ELA Support,2026-09-30,2027-06-30,14:30:00,16:30:00,12,Y,<NA>,N,79


In [305]:
# 6. Process Gary's report and separate records needing review

# Clear previous results so a failed run cannot reuse stale data
missing_attendance_summary = None
attendance_date_review = None
site_tables = {}

with pd.ExcelFile(missing_attendance_path) as workbook:
    missing_sheet_name = next(
        (
            name for name in workbook.sheet_names
            if "Missing Attendance" in name
        ),
        None
    )

if missing_sheet_name is None:
    raise ValueError("Missing Attendance report sheet not found.")

df_missing_att = read_external_report(
    missing_attendance_path,
    missing_sheet_name,
    base_keys + ["Date"]
)

# Validate identifiers
missing_keys = (
    df_missing_att[base_keys].isna().any(axis=1)
    | df_missing_att[base_keys].eq("").any(axis=1)
)

if missing_keys.any():
    display(df_missing_att.loc[missing_keys])
    raise ValueError("Missing session identifiers in Gary's report.")

# Validate dates
df_missing_att["Date"] = pd.to_datetime(
    df_missing_att["Date"], errors="coerce"
).dt.normalize()

if df_missing_att["Date"].isna().any():
    display(
        df_missing_att.loc[df_missing_att["Date"].isna()]
    )
    raise ValueError("Invalid or missing dates in Gary's report.")

df_missing_att = (
    df_missing_att
    .drop_duplicates(base_keys + ["Date"])
    .reset_index(drop=True)
)

df_missing_att["_missing_row_id"] = np.arange(len(df_missing_att))

# Include start time in the session identity
session_windows = (
    df_act[session_keys + ["Session End Date"]]
    .drop_duplicates()
    .copy()
)

for col in ["Session Start Date", "Session End Date"]:
    session_windows[col] = pd.to_datetime(
        session_windows[col], errors="coerce"
    ).dt.normalize()

candidates = df_missing_att.merge(
    session_windows,
    on=base_keys,
    how="left"
)

within_session = (
    candidates["Session Start Date"].notna()
    & candidates["Date"].ge(candidates["Session Start Date"])
    & (
        candidates["Session End Date"].isna()
        | candidates["Date"].le(candidates["Session End Date"])
    )
)

matched_candidates = candidates.loc[within_session].copy()

# Number of dated/timed sessions matching each missing-date record
match_counts = matched_candidates.groupby("_missing_row_id").size()

record_match_counts = (
    df_missing_att["_missing_row_id"]
    .map(match_counts)
    .fillna(0)
    .astype(int)
)

unmatched_ids = df_missing_att.loc[
    record_match_counts.eq(0), "_missing_row_id"
]

ambiguous_ids = df_missing_att.loc[
    record_match_counts.gt(1), "_missing_row_id"
]

# Unmatched dates: show the candidate windows for investigation
unmatched_review = candidates.loc[
    candidates["_missing_row_id"].isin(unmatched_ids)
].copy()

unmatched_review["Review Reason"] = np.where(
    unmatched_review["Session Start Date"].isna(),
    "No matching session or missing session start date",
    "Attendance date outside session start and end dates"
)

# Ambiguous dates: show only the windows that actually matched
ambiguous_review = matched_candidates.loc[
    matched_candidates["_missing_row_id"].isin(ambiguous_ids)
].copy()

ambiguous_review["Review Reason"] = (
    "Multiple matching session windows; Gary's report has no start time"
)

review_columns = base_keys + [
    "Date",
    "Session Start Date",
    "Session End Date",
    "Session Start Time",
    "Review Reason",
]

attendance_date_review = (
    pd.concat(
        [unmatched_review, ambiguous_review],
        ignore_index=True
    )[review_columns]
    .drop_duplicates()
    .sort_values(base_keys + ["Date", "Session Start Date"])
    .reset_index(drop=True)
)

# Only uniquely assigned records contribute to counts and percentages
valid_ids = df_missing_att.loc[
    record_match_counts.eq(1), "_missing_row_id"
]

matched_missing = matched_candidates.loc[
    matched_candidates["_missing_row_id"].isin(valid_ids)
].copy()

matched_missing = matched_missing.sort_values(
    session_keys + ["Date"]
)

matched_missing["Date Text"] = (
    matched_missing["Date"].dt.strftime("%m/%d/%Y")
)

missing_attendance_summary = (
    matched_missing.groupby(session_keys, dropna=False)
    .agg(
        **{
            "Days Missing Attendance": ("Date", "nunique"),
            "Dates Missing Attendance": (
                "Date Text",
                lambda dates: "; ".join(pd.unique(dates))
            ),
        }
    )
    .reset_index()
)

missing_attendance_summary["Days Missing Attendance"] = (
    missing_attendance_summary["Days Missing Attendance"]
    .astype("Int64")
)

print(
    f"Included {len(valid_ids)} missing-date records; "
    f"excluded {len(unmatched_ids)} unmatched and "
    f"{len(ambiguous_ids)} ambiguous records."
)

display(missing_attendance_summary.head())

if not attendance_date_review.empty:
    print("Excluded records requiring review:")
    display(attendance_date_review)

/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Included 305 missing-date records; excluded 0 unmatched and 58 ambiguous records.


,Site,Activity,Session,Session Start Date,Session Start Time,Days Missing Attendance,Dates Missing Attendance
0,High School for Media and Communications,Arts,Dominos,2026-10-02,16:30:00,5,10/02/2026; 10/09/2026; 10/16/2026; 10/23/2026...
1,High School for Media and Communications,Arts,Flower Design,2026-09-29,15:00:00,3,10/27/2026; 10/30/2026; 11/03/2026
2,High School for Media and Communications,Arts,Media Club,2026-09-28,14:30:00,5,10/20/2026; 10/23/2026; 10/27/2026; 10/30/2026...
3,High School for Media and Communications,Arts,Printmaking,2026-09-28,15:00:00,5,10/02/2026; 10/16/2026; 10/27/2026; 10/30/2026...
4,High School for Media and Communications,ELA and Math enrichment activities,ELA Support,2026-09-30,14:30:00,9,09/30/2026; 10/01/2026; 10/08/2026; 10/15/2026...


Excluded records requiring review:


,Site,Activity,Session,Date,Session Start Date,Session End Date,Session Start Time,Review Reason
0,The William Lynch School,Academic Enrichment,Tutoring Club (6th-8th Grade),2026-09-18,2026-09-14,2027-06-15,14:50:00,Multiple matching session windows; Gary's repo...
1,The William Lynch School,Academic Enrichment,Tutoring Club (6th-8th Grade),2026-09-18,2026-09-14,2027-06-15,15:30:00,Multiple matching session windows; Gary's repo...
2,The William Lynch School,Academic Enrichment,Tutoring Club (6th-8th Grade),2026-09-25,2026-09-14,2027-06-15,14:50:00,Multiple matching session windows; Gary's repo...
3,The William Lynch School,Academic Enrichment,Tutoring Club (6th-8th Grade),2026-09-25,2026-09-14,2027-06-15,15:30:00,Multiple matching session windows; Gary's repo...
4,The William Lynch School,Academic Enrichment,Tutoring Club (6th-8th Grade),2026-10-02,2026-09-14,2027-06-15,14:50:00,Multiple matching session windows; Gary's repo...
...,...,...,...,...,...,...,...,...
111,The William Lynch School,"Sports, Fitness & Wellness",Active Sports (6th-8th Grade),2026-10-30,2026-09-14,2027-06-15,16:15:00,Multiple matching session windows; Gary's repo...
112,The William Lynch School,"Sports, Fitness & Wellness",Active Sports (6th-8th Grade),2026-11-02,2026-09-14,2027-06-15,15:30:00,Multiple matching session windows; Gary's repo...
113,The William Lynch School,"Sports, Fitness & Wellness",Active Sports (6th-8th Grade),2026-11-02,2026-09-14,2027-06-15,16:15:00,Multiple matching session windows; Gary's repo...
114,The William Lynch School,"Sports, Fitness & Wellness",Active Sports (6th-8th Grade),2026-11-05,2026-09-14,2027-06-15,15:30:00,Multiple matching session windows; Gary's repo...


In [307]:
# 7. Create one Site Summary table for every grant site

site_tables = {}

sites = (
    site_master["Site"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

output_columns = [
    "Site",
    "Activity",
    "Session",
    "Session Start Date",
    "Session End Date",
    "Session Start Time",
    "Session End Time",
    "Days Scheduled",
    "Enrolled Participant",
    "Average Daily Attendance",
    "Missing Attendance (%)",
    "Staff Assigned",
    "Partner Assigned",
    "Days Scheduled to Date",
    "Days Missing Attendance to Date",
    "Dates Missing Attendance",
    "Partner",
]


# Check whether enrollment or attendance records do not match
# any Activity-Session Details record
known_base_sessions = (
    df_act[base_keys]
    .drop_duplicates()
)

for report_name, report_df in [
    (
        "Session Enrollment by Session",
        df_enr,
    ),
    (
        "Daily Activity Attendance Summary",
        df_att,
    ),
]:
    unmatched_report = report_df.merge(
        known_base_sessions,
        on=base_keys,
        how="left",
        indicator=True,
    )

    unmatched_report = unmatched_report.loc[
        unmatched_report["_merge"].eq("left_only"),
        base_keys,
    ]

    if not unmatched_report.empty:
        print(
            f"{report_name} records without matching "
            f"Activity-Session Details:"
        )

        display(
            unmatched_report
            .drop_duplicates()
            .sort_values(base_keys)
        )


for site in sites:
    # Pull the site's data from each report
    site_act = df_act.loc[
        df_act["Site"].eq(site)
    ].copy()

    site_enr = df_enr.loc[
        df_enr["Site"].eq(site)
    ].copy()

    site_att = df_att.loc[
        df_att["Site"].eq(site)
    ].copy()

    site_missing = (
        missing_attendance_summary.loc[
            missing_attendance_summary[
                "Site"
            ].eq(site)
        ]
        .copy()
    )

    # If the site has no Activity-Session Details records,
    # create an empty table with the required output columns
    if site_act.empty:
        site_tables[site] = pd.DataFrame(
            columns=output_columns
        )

        print(
            f"Warning: No Activity-Session Details "
            f"records were found for {site}."
        )

        continue


    # Activity-Session Details is the authoritative table
    # because it contains the session start and end dates.
    #
    # Enrollment and attendance do not contain start dates.
    # Their values are therefore matched using Site,
    # Activity, and Session.
    merged_site = site_act.merge(
        site_enr,
        on=base_keys,
        how="left",
        validate="many_to_one",
    )

    merged_site = merged_site.merge(
        site_att,
        on=base_keys,
        how="left",
        validate="many_to_one",
    )


    # Missing-attendance records have already been assigned
    # to the appropriate dated session in Cell 6.
    merged_site = merged_site.merge(
        site_missing,
        on=session_keys,
        how="left",
        validate="one_to_one",
    )


    # Exclude sessions that begin today or later.
    # Retain records with an unknown start date.
    merged_site = merged_site.loc[
        merged_site[
            "Session Start Date"
        ].isna()
        | merged_site[
            "Session Start Date"
        ].lt(today)
    ].copy()


    # No matching Gary's-report record means that no
    # missing-attendance dates were reported for the session.
    merged_site[
        "Days Missing Attendance"
    ] = (
        merged_site[
            "Days Missing Attendance"
        ]
        .fillna(0)
        .astype("Int64")
    )

    merged_site[
        "Dates Missing Attendance"
    ] = (
        merged_site[
            "Dates Missing Attendance"
        ]
        .fillna("-")
    )


    # Fill missing staff and partner indicators
    for col in [
        "Staff Assigned",
        "Partner Assigned",
    ]:
        merged_site[col] = (
            merged_site[col]
            .fillna("N")
        )


    # Convert scheduled-day columns to whole numbers
    for col in [
        "Days Scheduled",
        "Days Scheduled to Date",
    ]:
        merged_site[col] = (
            pd.to_numeric(
                merged_site[col],
                errors="coerce",
            )
            .astype("Int64")
        )


    # Calculate missing-attendance percentage using
    # scheduled days through the report date
    denominator = (
        merged_site[
            "Days Scheduled to Date"
        ]
    )

    missing_percentage = (
        merged_site[
            "Days Missing Attendance"
        ]
        .div(
            denominator.where(
                denominator > 0
            )
        )
        .mul(100)
    )

    merged_site[
        "Missing Attendance (%)"
    ] = (
        missing_percentage
        .round()
        .astype("Int64")
        .astype("string")
        .add("%")
        .fillna("-")
    )


    # Flag records where the missing-day count exceeds
    # the scheduled-day count
    inconsistent = (
        merged_site[
            "Days Missing Attendance"
        ]
        .gt(
            merged_site[
                "Days Scheduled to Date"
            ]
        )
        .fillna(False)
    )

    if inconsistent.any():
        print(
            f"Check missing-attendance counts for {site}:"
        )

        display(
            merged_site.loc[
                inconsistent,
                session_keys
                + [
                    "Days Scheduled to Date",
                    "Days Missing Attendance",
                ],
            ]
        )


    # Rename the missing-attendance count for the output
    merged_site = merged_site.rename(
        columns={
            "Days Missing Attendance":
                "Days Missing Attendance to Date",
        }
    )


    # Sort while the date columns are still datetime values
    merged_site = (
        merged_site
        .sort_values(
            [
                "Activity",
                "Session",
                "Session Start Date",
                "Session Start Time",
            ],
            na_position="last",
        )
        .reset_index(drop=True)
    )


    # Display start and end dates without time values
    for col in [
        "Session Start Date",
        "Session End Date",
    ]:
        merged_site[col] = (
            pd.to_datetime(
                merged_site[col],
                errors="coerce",
            )
            .dt.strftime("%m/%d/%Y")
        )


    # Select and arrange final output columns
    merged_site = (
        merged_site[
            output_columns
        ]
        .copy()
    )


    # Replace remaining missing values with "-"
    merged_site = (
        merged_site
        .astype(object)
        .where(
            merged_site.notna(),
            "-",
        )
    )


    site_tables[site] = merged_site

    for col in ["Session Start Time", "Session End Time"]:
        merged_site[col] = merged_site[col].map(
            lambda value: (
                datetime.strptime(value, "%H:%M:%S")
                .strftime("%I:%M:%S %p")
                .lstrip("0")
            )
            if pd.notna(value)
            else "-"
        )

# Preview the summary table for every grant site
for site_name in sites:
    final_df = site_tables[site_name]

    print(
        f"\nSite: {site_name} "
        f"({len(final_df)} sessions)"
    )

    display(
        final_df.head()
    )


print(
    f"\nCreated Site Summary tables for "
    f"{len(site_tables)} grant sites."
)


Site: High School for Media and Communications (20 sessions)


,Site,Activity,Session,Session Start Date,Session End Date,Session Start Time,Session End Time,Days Scheduled,Enrolled Participant,Average Daily Attendance,Missing Attendance (%),Staff Assigned,Partner Assigned,Days Scheduled to Date,Days Missing Attendance to Date,Dates Missing Attendance,Partner
0,High School for Media and Communications,Arts,Dominos,10/02/2026,05/28/2027,4:30:00 PM,5:30:00 PM,36,4.0,4,83%,Y,Y,6,5,10/02/2026; 10/09/2026; 10/16/2026; 10/23/2026...,Arete
1,High School for Media and Communications,Arts,Flower Design,09/29/2026,05/28/2027,3:00:00 PM,5:00:00 PM,70,15.0,7,27%,Y,Y,11,3,10/27/2026; 10/30/2026; 11/03/2026,Arete
2,High School for Media and Communications,Arts,Media Club,09/28/2026,05/28/2027,2:30:00 PM,3:30:00 PM,71,18.0,5,42%,N,N,12,5,10/20/2026; 10/23/2026; 10/27/2026; 10/30/2026...,-
3,High School for Media and Communications,Arts,Printmaking,09/28/2026,05/28/2027,3:00:00 PM,5:00:00 PM,70,2.0,2,45%,Y,N,11,5,10/02/2026; 10/16/2026; 10/27/2026; 10/30/2026...,-
4,High School for Media and Communications,ELA and Math enrichment activities,ELA Support,09/30/2026,06/30/2027,2:30:00 PM,4:30:00 PM,79,7.0,2,75%,Y,N,12,9,09/30/2026; 10/01/2026; 10/08/2026; 10/15/2026...,-



Site: P.S. 28 Wright Brothers (82 sessions)


,Site,Activity,Session,Session Start Date,Session End Date,Session Start Time,Session End Time,Days Scheduled,Enrolled Participant,Average Daily Attendance,Missing Attendance (%),Staff Assigned,Partner Assigned,Days Scheduled to Date,Days Missing Attendance to Date,Dates Missing Attendance,Partner
0,P.S. 28 Wright Brothers,Arts,Arts & Crafts 1A,10/14/2026,06/18/2027,4:30:00 PM,6:00:00 PM,34,14,13,0%,N,Y,3,0,-,ACDP – Community Association of Progressive Do...
1,P.S. 28 Wright Brothers,Arts,Arts & Crafts 3A,10/14/2026,06/18/2027,4:30:00 PM,6:00:00 PM,33,9,8,0%,N,Y,2,0,-,ACDP – Community Association of Progressive Do...
2,P.S. 28 Wright Brothers,Arts,Arts & Crafts 3B,10/14/2026,06/18/2027,3:00:00 PM,4:30:00 PM,33,9,8,0%,N,Y,2,0,-,ACDP – Community Association of Progressive Do...
3,P.S. 28 Wright Brothers,Arts,Arts & Crafts 4A,10/14/2026,06/18/2027,4:30:00 PM,6:00:00 PM,32,8,7,0%,N,Y,3,0,-,ACDP – Community Association of Progressive Do...
4,P.S. 28 Wright Brothers,Arts,Arts & Crafts 4B,10/14/2026,06/18/2027,3:00:00 PM,4:30:00 PM,32,9,9,0%,N,Y,3,0,-,ACDP – Community Association of Progressive Do...



Site: PS 5 Ellen Lurie (34 sessions)


,Site,Activity,Session,Session Start Date,Session End Date,Session Start Time,Session End Time,Days Scheduled,Enrolled Participant,Average Daily Attendance,Missing Attendance (%),Staff Assigned,Partner Assigned,Days Scheduled to Date,Days Missing Attendance to Date,Dates Missing Attendance,Partner
0,PS 5 Ellen Lurie,Academic Enrichment,iReady - 2nd Grade,10/05/2026,06/11/2027,2:30:00 PM,3:30:00 PM,29,18,18,0%,Y,Y,3,0,-,PS 5 Ellen Lurie
1,PS 5 Ellen Lurie,Academic Enrichment,iReady - 3rd Grade,10/05/2026,06/11/2027,3:30:00 PM,4:30:00 PM,29,14,12,0%,Y,Y,3,0,-,PS 5 Ellen Lurie
2,PS 5 Ellen Lurie,Academic Enrichment,iReady - 4th Grade,10/05/2026,06/11/2027,3:30:00 PM,4:30:00 PM,28,17,16,0%,Y,Y,4,0,-,PS 5 Ellen Lurie
3,PS 5 Ellen Lurie,Academic Enrichment,iReady - 5th Grade,10/05/2026,06/11/2027,2:30:00 PM,3:30:00 PM,28,20,20,0%,Y,Y,4,0,-,PS 5 Ellen Lurie
4,PS 5 Ellen Lurie,Academic Enrichment,iReady - Kindergarten & First Grade,10/05/2026,06/11/2027,4:30:00 PM,5:30:00 PM,29,19,17,0%,Y,Y,3,0,-,PS 5 Ellen Lurie



Site: The William Lynch School (37 sessions)


,Site,Activity,Session,Session Start Date,Session End Date,Session Start Time,Session End Time,Days Scheduled,Enrolled Participant,Average Daily Attendance,Missing Attendance (%),Staff Assigned,Partner Assigned,Days Scheduled to Date,Days Missing Attendance to Date,Dates Missing Attendance,Partner
0,The William Lynch School,Academic Enrichment,Debate (6th-8th Grade),09/14/2026,06/15/2027,3:30:00 PM,4:15:00 PM,34,39,17,12%,Y,Y,8,1,11/04/2026,MMCC - Mosholu Montefiore Community Center
1,The William Lynch School,Academic Enrichment,Literacy Club ( K- 1st Grade),09/14/2026,06/15/2027,2:50:00 PM,3:30:00 PM,97,28,20,25%,Y,Y,20,5,10/23/2026; 10/26/2026; 10/27/2026; 10/30/2026...,MMCC - Mosholu Montefiore Community Center
2,The William Lynch School,Academic Enrichment,Literacy Club (2nd -3rd Grade),09/14/2026,06/15/2027,3:30:00 PM,4:15:00 PM,66,19,16,25%,Y,Y,16,4,10/21/2026; 10/29/2026; 11/04/2026; 11/05/2026,MMCC - Mosholu Montefiore Community Center
3,The William Lynch School,Academic Enrichment,Mini Debate ( K-1st Grade),09/14/2026,06/15/2027,3:30:00 PM,4:15:00 PM,31,28,18,29%,Y,Y,7,2,10/23/2026; 10/30/2026,MMCC - Mosholu Montefiore Community Center
4,The William Lynch School,Academic Enrichment,Movie & Discussion ( K- 1st Grade),09/14/2026,06/15/2027,4:15:00 PM,5:00:00 PM,31,28,-,100%,Y,Y,7,7,09/18/2026; 09/25/2026; 10/02/2026; 10/09/2026...,MMCC - Mosholu Montefiore Community Center



Created Site Summary tables for 4 grant sites.


## 🌈 Color Coding and Full Report Download

In [308]:
# Color Coding
from openpyxl import load_workbook
from openpyxl.styles import PatternFill


def clean_sheet_name(name):
    name = str(name).strip()

    for character in [":", "/", "\\", "?", "*", "[", "]"]:
        name = name.replace(character, "")

    return name.strip("'")[:31] or "Sheet"


def apply_missing_highlights(
    output_filename,
    total_missing_rows,
    flag_cols,
    young_dob_rows,
    staff_missing_rows=None,
    staff_flag_cols=None,
    site_tables=None
):
    red_fill = PatternFill(
        "solid", start_color="FF9999", end_color="FF9999"
    )
    blue_fill = PatternFill(
        "solid", start_color="9999FF", end_color="9999FF"
    )

    wb = load_workbook(output_filename)

    # Student required-information pull-outs
    flag_to_original = {
        flag.removesuffix("_missing"): flag
        for flag in flag_cols
        if flag.endswith("_missing")
        and flag != "State ParticipantID_missing"
    }

    for site, group in total_missing_rows.groupby("Site"):
        ws = wb[clean_sheet_name("Missing - " + str(site))]
        headers = {cell.value: cell.column for cell in ws[1]}

        for row_idx, (_, record) in enumerate(
            group.iterrows(), start=2
        ):
            for original_col, flag_col in flag_to_original.items():
                if (
                    original_col in headers
                    and record.get(flag_col, 0) == 1
                ):
                    ws.cell(
                        row=row_idx,
                        column=headers[original_col]
                    ).fill = red_fill

    # Staff required-information pull-out
    if staff_missing_rows is not None and staff_flag_cols is not None:
        ws = wb["Pull out - Missing Staff Info"]
        headers = {cell.value: cell.column for cell in ws[1]}

        staff_mapping = {
            "Staff Type_missing": "Staff Type",
            "Employment Type_missing": "Compensation Type",
            "Funded by 21st CCLC_missing": "Funder",
        }

        for row_idx, (_, record) in enumerate(
            staff_missing_rows.iterrows(), start=2
        ):
            for flag_col in staff_flag_cols:
                original_col = staff_mapping[flag_col]

                if (
                    original_col in headers
                    and record.get(flag_col, 0) == 1
                ):
                    ws.cell(
                        row=row_idx,
                        column=headers[original_col]
                    ).fill = red_fill

    # Atypical age review
    ws = wb["DOB - Atypical Age Check"]
    headers = {cell.value: cell.column for cell in ws[1]}

    if "Date Of Birth" in headers:
        for row_idx in range(2, ws.max_row + 1):
            ws.cell(
                row=row_idx,
                column=headers["Date Of Birth"]
            ).fill = blue_fill

    # Site summary tables
    if site_tables is not None:
        for site_name in site_tables:
            ws = wb[clean_sheet_name(site_name)]
            headers = {cell.value: cell.column for cell in ws[1]}

            for row in ws.iter_rows(min_row=2):
                row_idx = row[0].row

                # Missing values and unassigned staff/partners
                for cell in row:
                    column_name = ws.cell(
                        row=1, column=cell.column
                    ).value

                    is_blank = (
                        cell.value is None
                        or str(cell.value).strip() in ["", "-"]
                    )

                    assignment_missing = (
                        column_name in [
                            "Staff Assigned",
                            "Partner Assigned",
                        ]
                        and str(cell.value).strip().upper() == "N"
                    )

                    # An empty missing-date list is not an issue
                    if (
                        is_blank
                        and column_name != "Dates Missing Attendance"
                    ) or assignment_missing:
                        cell.fill = red_fill

                # Highlight count, percentage, and dates when count > 0
                count_col = "Days Missing Attendance to Date"

                if count_col in headers:
                    count = ws.cell(
                        row=row_idx,
                        column=headers[count_col]
                    ).value

                    try:
                        has_missing_attendance = float(count) > 0
                    except (TypeError, ValueError):
                        has_missing_attendance = False

                    if has_missing_attendance:
                        for column_name in [
                            count_col,
                            "Missing Attendance (%)",
                            "Dates Missing Attendance",
                        ]:
                            if column_name in headers:
                                ws.cell(
                                    row=row_idx,
                                    column=headers[column_name]
                                ).fill = red_fill

            # Date display and widths
            for column_name in [
                "Session Start Date",
                "Session End Date",
            ]:
                if column_name in headers:
                    col_idx = headers[column_name]

                    ws.column_dimensions[
                        ws.cell(1, col_idx).column_letter
                    ].width = 18

                    for row_idx in range(2, ws.max_row + 1):
                        ws.cell(
                            row=row_idx,
                            column=col_idx
                        ).number_format = "mm/dd/yyyy"

    wb.save(output_filename)
    wb.close()

    print("Highlights applied.")

In [309]:
# Test output
from IPython.display import FileLink, display
import os

output_filename = "WeeklyUpdates_test_CSD6_1105.xlsx"

# Hide internal checking columns
internal_columns = {
    "DOB_too_young",
    "DOB_age_grade_review",
    "_staff_id",
}

hide_cols = {
    "Race/Ethnicity",
    "English Learner Status",
    "Lunch Status",
    "Special Education Status",
    "IDEA Disability Type",
}

student_display_cols = [
    col for col in total_missing_rows.columns
    if not col.endswith("_missing")
    and col not in internal_columns
    and col not in hide_cols
]

dob_display_cols = [
    col for col in young_dob_rows.columns
    if not col.endswith("_missing")
    and col not in internal_columns
]

staff_display_cols = [
    col for col in staff_missing_rows.columns
    if not col.endswith("_missing")
    and col not in internal_columns
]

# Check for sheet-name collisions before writing
sheet_names = [
    "Student Summary Statistics",
    "Family Component Summary",
    "Missing Student Summary",
    "DOB - Atypical Age Check",
    "Missing Staff Summary",
    "Pull out - Missing Staff Info",
    "Attendance Date Review",
]

if not missing_site_rows.empty:
    sheet_names.append("Pull out - Missing Site Info")

sheet_names += [
    clean_sheet_name("Missing - " + str(site))
    for site in total_missing_rows["Site"].dropna().unique()
]

sheet_names += [
    clean_sheet_name(site)
    for site in site_tables
]

if len(sheet_names) != len(set(name.lower() for name in sheet_names)):
    raise ValueError(
        "Some output sheet names become identical after cleaning "
        "or truncation. Resolve the names before exporting."
    )

with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:

    styled_totals.to_excel(
        writer,
        sheet_name="Student Summary Statistics",
        index=False
    )

    result.to_excel(
        writer,
        sheet_name="Family Component Summary",
        index=False
    )

    missing_summary.to_excel(
        writer,
        sheet_name="Missing Student Summary",
        index=False
    )

    # Keep these records visible if any students lack a site
    if not missing_site_rows.empty:
        missing_site_display_cols = [
            col for col in missing_site_rows.columns
            if not col.endswith("_missing")
            and col not in internal_columns
            and col not in hide_cols
        ]

        missing_site_rows[missing_site_display_cols].to_excel(
            writer,
            sheet_name="Pull out - Missing Site Info",
            index=False
        )

    # Required-information pull-out for each site with flagged students
    for site, group in total_missing_rows.groupby("Site"):
        group[student_display_cols].to_excel(
            writer,
            sheet_name=clean_sheet_name("Missing - " + str(site)),
            index=False
        )

    young_dob_rows[dob_display_cols].to_excel(
        writer,
        sheet_name="DOB - Atypical Age Check",
        index=False
    )

    staff_missing_summary.to_excel(
        writer,
        sheet_name="Missing Staff Summary",
        index=False
    )

    staff_missing_rows[staff_display_cols].to_excel(
        writer,
        sheet_name="Pull out - Missing Staff Info",
        index=False
    )

    # One session sheet for every master-list site
    for site_name, final_df in site_tables.items():
        final_df.to_excel(
            writer,
            sheet_name=clean_sheet_name(site_name),
            index=False
        )

    attendance_date_review.to_excel(
        writer,
        sheet_name="Attendance Date Review",
        index=False
    )

    

# Apply formatting only after ExcelWriter has closed
apply_missing_highlights(
    output_filename,
    total_missing_rows,
    flag_cols,
    young_dob_rows,
    staff_missing_rows=staff_missing_rows,
    staff_flag_cols=staff_flag_cols,
    site_tables=site_tables
)


print(f"Workbook saved to:\n{os.path.abspath(output_filename)}")

display(FileLink(output_filename))

Highlights applied.
Workbook saved to:
/Users/lgteam/Desktop/Weekly Updates Automation Tool/WeeklyUpdates_test_CSD6_1105.xlsx


/Users/lgteam/Desktop/Weekly Updates Automation Tool/WeeklyUpdates_test_CSD6_1105.xlsx

In [310]:
# To be removed
# Test output

# Export the complete workbook without color coding

import os
import pandas as pd
from IPython.display import FileLink, display


output_filename = (
    "/Users/lgteam/Desktop/Weekly Updates Automation Tool/"
    "WeeklyUpdates_CFL_Nov_2_2026.xlsx"
)


def clean_sheet_name(name):
    name = str(name).strip()

    for character in [
        ":",
        "/",
        "\\",
        "?",
        "*",
        "[",
        "]",
    ]:
        name = name.replace(
            character,
            "",
        )

    return (
        name.strip("'")[:31]
        or "Sheet"
    )


# Columns displayed in student pull-out sheets
internal_columns = {
    "DOB_too_young",
    "DOB_age_grade_review",
    "_staff_id",
}

hide_cols = {
    "Race/Ethnicity",
    "English Learner Status",
    "Lunch Status",
    "Special Education Status",
    "IDEA Disability Type",
}

student_display_cols = [
    col
    for col in total_missing_rows.columns
    if not col.endswith("_missing")
    and col not in internal_columns
    and col not in hide_cols
]

dob_display_cols = [
    col
    for col in young_dob_rows.columns
    if not col.endswith("_missing")
    and col not in internal_columns
]

staff_display_cols = [
    col
    for col in staff_missing_rows.columns
    if not col.endswith("_missing")
    and col not in internal_columns
]


with pd.ExcelWriter(
    output_filename,
    engine="openpyxl",
) as writer:

    # Student summary
    styled_totals.to_excel(
        writer,
        sheet_name="Student Summary Statistics",
        index=False,
    )

    # Family component
    result.to_excel(
        writer,
        sheet_name="Family Component Summary",
        index=False,
    )

    # Missing student summary
    missing_summary.to_excel(
        writer,
        sheet_name="Missing Student Summary",
        index=False,
    )

    # Students missing site information
    if not missing_site_rows.empty:
        missing_site_display_cols = [
            col
            for col in missing_site_rows.columns
            if not col.endswith("_missing")
            and col not in internal_columns
            and col not in hide_cols
        ]

        missing_site_rows[
            missing_site_display_cols
        ].to_excel(
            writer,
            sheet_name="Pull out - Missing Site Info",
            index=False,
        )

    # Individual missing-information sheet for each site
    for site, group in (
        total_missing_rows.groupby(
            "Site"
        )
    ):
        group[
            student_display_cols
        ].to_excel(
            writer,
            sheet_name=clean_sheet_name(
                "Missing - " + str(site)
            ),
            index=False,
        )

    # Atypical DOB/grade records
    young_dob_rows[
        dob_display_cols
    ].to_excel(
        writer,
        sheet_name="DOB - Atypical Age Check",
        index=False,
    )

    # Missing staff summary
    staff_missing_summary.to_excel(
        writer,
        sheet_name="Missing Staff Summary",
        index=False,
    )

    # Missing staff details
    staff_missing_rows[
        staff_display_cols
    ].to_excel(
        writer,
        sheet_name="Pull out - Missing Staff Info",
        index=False,
    )

    # Site session summary tables
    for site_name, final_df in (
        site_tables.items()
    ):
        sheet_name = clean_sheet_name(
            site_name
        )

        print(
            f"Writing site sheet: "
            f"{repr(sheet_name)}"
        )

        final_df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False,
        )


print(
    "Workbook exported successfully:"
)

print(
    os.path.abspath(
        output_filename
    )
)

display(
    FileLink(
        output_filename
    )
)

Writing site sheet: 'High School for Media and Commu'
Writing site sheet: 'P.S. 28 Wright Brothers'
Writing site sheet: 'PS 5 Ellen Lurie'
Writing site sheet: 'The William Lynch School'
Workbook exported successfully:
/Users/lgteam/Desktop/Weekly Updates Automation Tool/WeeklyUpdates_CFL_Nov_2_2026.xlsx


/Users/lgteam/Desktop/Weekly Updates Automation Tool/WeeklyUpdates_CFL_Nov_2_2026.xlsx